In [ ]:
# !pip install matplotlib numpy pandas wandb x-transformers nbconvert ipykernel datasets tqdm
# !pip install ipywidgets --upgrade

In [ ]:
# !sudo apt install unzip -y

# !git clone https://github.com/fchollet/ARC.git
# !git clone https://github.com/michaelhodel/re-arc.git
# !cd re-arc && unzip re_arc.zip

# !rm -rf re-arc/.git
# !rm -rf ARC/.git

In [ ]:
# !pip install gdown
# !gdown --folder https://drive.google.com/drive/folders/1LIc90R2MYaY_2ErAYywPZJP-SFspZt44 -O ./arc-prize-2025

In [ ]:
import sys
print(sys.version)

In [ ]:
import os
from os.path import join as path_join
import json
from pathlib import Path
from functools import reduce
import matplotlib.pyplot as plt
from matplotlib import colors
from mpl_toolkits.axes_grid1 import ImageGrid
import matplotlib.patches as patches
import io
import time

import numpy as np
import pandas as pd
from PIL import Image
import datetime
import math

In [ ]:
import torch
import torch.nn.functional as F
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torch.optim.lr_scheduler import LambdaLR

torch.set_float32_matmul_precision("high")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
from utils import *

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def prep_target_seq(input):
    # Create a new tensor with the same shape
    shifted_input = torch.empty_like(input)

    # Set the first token of every sequence to the start token
    shifted_input[:, 0] = SOS_TOKEN

    # Copy all tokens except the last one from input_examples to shifted_input_examples starting at position 1
    shifted_input[:, 1:] = input[:, :-1]

    return shifted_input

#### wanb

In [ ]:
import wandb
# https://docs.wandb.ai/guides/track/environment-variables
import os

WANDB_API_KEY = 'a371fb004de1c0ca0ad9911ca48782eec901bf52'
os.environ['WANDB_API_KEY'] = WANDB_API_KEY
os.environ['WANDB_MODE'] = 'online' #'offline'
wandb.login()

#### Datasets

In [ ]:
data_path = Path('ARC/data')
training_path = data_path / 'training'
evaluation_path = data_path / 'evaluation'

In [ ]:
# size is max 30x30
# integer between 0 and 9, which are visualized as colors

def store_tasks(path, is_train = True):
  training_tasks = sorted(os.listdir(path))
  records = [] 
  for task_name in training_tasks:
    task_file = str(path / task_name)

    with open(task_file, 'r') as f:
      task = json.load(f)

      for key in task:
        if key not in ['train', 'test']:
          continue
        for t in task[key]:
          rec = {
            'task_id' : task_name, 
            'input' : t['input'], 
            'output' : t['output'],
            'final_task': key != 'train',
            'train' : is_train 
          }
          records.append(rec)

  #df = pd.DataFrame(columns=['task_id', 'input', 'output', 'final_task', 'train']) 
  df = pd.DataFrame(records)
  
  return df


df_train = store_tasks(training_path, True) 
df_test = store_tasks(evaluation_path, False)
df = pd.concat([df_train, df_test])

train_task_ids = df_train['task_id'].unique()
test_task_ids = df_test['task_id'].unique()

In [ ]:
def load_json(path):
    with open(path, 'r') as f:
        return json.load(f)
    
training_challenges_path = 'arc-prize-2025/arc-agi_training_challenges.json'
training_solutions_path = 'arc-prize-2025/arc-agi_training_solutions.json'

evaluation_challenges_path = 'arc-prize-2025/arc-agi_evaluation_challenges.json'
evaluation_solutions_path = 'arc-prize-2025/arc-agi_evaluation_solutions.json'

training_challenges = load_json(training_challenges_path)
training_solutions = load_json(training_solutions_path)

evaluation_challenges = load_json(evaluation_challenges_path)
evaluation_solutions = load_json(evaluation_solutions_path)

#print(len(training_challenges), len(training_solutions), len(evaluation_challenges), len(evaluation_solutions))

def create_arc2_dataset(challenges, solutions):
    records = []
    for key, value in challenges.items():
        # save examples
        for example in value['train']:
            rec = {
                'task_id' : key, 
                'input' : example['input'], 
                'output' : example['output'],
                'final_task': False,
            }
            records.append(rec)

        # save final task
        for task_input, task_output in zip(value['test'], solutions[key]):
            rec = {
                'task_id' : key, 
                'input' : task_input['input'], 
                'output' : task_output,
                'final_task': True,
            }
            records.append(rec)

    return records

df_train_arc2 = pd.DataFrame(create_arc2_dataset(training_challenges, training_solutions))
df_test_arc2 = pd.DataFrame(create_arc2_dataset(evaluation_challenges, evaluation_solutions))

train_task_ids_arc2 = df_train_arc2['task_id'].unique()
test_task_ids_arc2 = df_test_arc2['task_id'].unique()

In [ ]:
NUMBER_OF_COLORS = 10 +1 +1 # integer between 0 and 9 = 10, and +1 my class (used as background (10)) + 1 for SOS token
MAX_IMAGE_SIZE = 32
MAX_SEQUENCE_SIZE = MAX_IMAGE_SIZE * MAX_IMAGE_SIZE
BACKGROUND_CLASS = 10
SOS_TOKEN = 11

# make all images 32 by 32 and fill empty spaces by additional class
def placeholder(arg, sizes = (MAX_IMAGE_SIZE, MAX_IMAGE_SIZE), background=BACKGROUND_CLASS):
  img = np.array(arg)
  holder = np.full(sizes, background, dtype=int)
  holder[0:img.shape[0], 0:img.shape[1]] = img

  return holder

In [ ]:
class CustomDatasetTestTransformationsDiscrete(Dataset):
    def __init__(self, task_ids, df, n_of_samples=2):
        self.task_ids = task_ids
        self.data_by_task_id = {
            task_id: df.query(f'task_id == "{task_id}"').to_dict(orient='records')
            for task_id in task_ids
        }
        self.n_of_samples = n_of_samples
        
    def __len__(self):
        return len(self.task_ids)

    def __getitem__(self, idx):
        task_id = self.task_ids[idx]
        examples = self.data_by_task_id[task_id]
        selected_examples = random.sample(examples, self.n_of_samples + 1)

        transformation_1 = sample_transformations()
        transformation_2 = lambda x: placeholder(transformation_1(x))
        transformation_32 = lambda x: torch.tensor(transformation_2(x).flatten(), dtype=torch.long)

        data_item_input = [transformation_32(example['input']) for example in selected_examples]
        data_item_output = [transformation_32(example['output']) for example in selected_examples]

        return [
            torch.stack(data_item_input[:self.n_of_samples]), torch.stack(data_item_output[:self.n_of_samples]), 
            torch.stack(data_item_input[self.n_of_samples:]), torch.stack(data_item_output[self.n_of_samples:]),
        ]
    

class CustomDatasetTestTransformationsAdaptDiscrete(Dataset):
    def __init__(self, task_ids, df, n_of_samples=2):
        self.task_ids = task_ids
        self.data_by_task_id = {
            task_id: df.query(f'task_id == "{task_id}"').to_dict(orient='records')
            for task_id in task_ids
        }
        self.n_of_samples = n_of_samples
        self.df = df
        
    def __len__(self):
        return len(self.task_ids)

    def __getitem__(self, idx):
        task_id = self.task_ids[idx]
        examples = self.df.query(f'final_task == False and task_id == "{task_id}"').to_dict(orient='records')
        selected_examples = random.sample(examples, self.n_of_samples)

        final_task = self.df.query(f'final_task == True and task_id == "{task_id}"').to_dict(orient='records')[0]

        transformation_1 = sample_transformations()
        transformation_2 = lambda x: placeholder(transformation_1(x))
        transformation_32 = lambda x: torch.tensor(transformation_2(x).flatten(), dtype=torch.long)

        data_item_input = [transformation_32(example['input']) for example in selected_examples]
        data_item_output = [transformation_32(example['output']) for example in selected_examples]
        data_item_input_d = [transformation_32(final_task['input'])]
        data_item_output_d = [transformation_32(final_task['output'])]

        return [
            torch.stack(data_item_input), torch.stack(data_item_output), 
            torch.stack(data_item_input_d), torch.stack(data_item_output_d)
        ]
    

class CustomDatasetTestTransformationsDiscreteReArc2(Dataset):
    def __init__(self, task_ids, data_points, n_of_samples=2):
        self.task_ids = task_ids
        self.data_points = data_points
        self.n_of_samples = n_of_samples
        self.task_data = self._load_all_tasks()
        
    def _load_all_tasks(self):
        """Preloads all task files into memory."""
        task_data = {}
        for task_id in self.task_ids:
            task_file = f're-arc/re_arc/tasks/{task_id}'
            with open(task_file, 'r') as f:
                examples = json.load(f)

                examples_filtered = []
                for example in examples:
                    if len(example['input']) <= 32 and len(example['input'][0]) <= 32 and len(example['output']) <= 32 and len(example['output'][0]) <= 32:
                        examples_filtered.append(example)

                task_data[task_id] = examples_filtered
        return task_data
    
    def __len__(self):
        return len(self.task_ids)

    def __getitem__(self, idx):
        task_id = self.task_ids[idx]
        examples = self.task_data[task_id]  # Fetch preloaded data
        
        selected_examples = random.sample(examples, self.n_of_samples+1)

        transformation_1 = sample_transformations()
        transformation_2 = lambda x: placeholder(transformation_1(x))
        transformation_32 = lambda x: torch.tensor(transformation_2(x).flatten(), dtype=torch.long)

        data_item_input = [transformation_32(example['input']) for example in selected_examples]
        data_item_output = [transformation_32(example['output']) for example in selected_examples]

        return [
            torch.stack(data_item_input[:self.n_of_samples]), torch.stack(data_item_output[:self.n_of_samples]), 
            torch.stack(data_item_input[self.n_of_samples:]), torch.stack(data_item_output[self.n_of_samples:]),
        ]

In [ ]:
# # make sure arc task is not in the re-arc dataset
# def get_arc_rearc_tasks(task_id):
#     task_file = f're-arc/re_arc/tasks/{task_id}'
#     with open(task_file, 'r') as f:
#         examples = json.load(f)

#         examples_filtered = []
#         for example in examples:
#             if len(example['input']) <= 32 and len(example['input'][0]) <= 32 and len(example['output']) <= 32 and len(example['output'][0]) <= 32:
#                 examples_filtered.append(example)
#     rearc_tasks = examples_filtered


#     data = df.query(f'task_id == "{task_id}"').to_dict(orient='records')
#     arc_tasks = []
#     for example in data:
#         arc_tasks.append({
#             'input': example['input'],
#             'output': example['output'],
#         })

#     return rearc_tasks, arc_tasks

# #task_id = '0a938d79.json'
# for task_id in train_task_ids:
#     rearc_tasks, arc_tasks = get_arc_rearc_tasks(task_id)
#     #print(task_id, len(rearc_tasks), len(arc_tasks))
#     for rearc_task in rearc_tasks:
#         for arc_task in arc_tasks:
#             if rearc_task['input'] == arc_task['input']:
#                 print(rearc_task['input'], rearc_task['output'])
#                 print(arc_task['input'], arc_task['output'])
#                 print('---')

In [ ]:
class CustomDatasetTestTransformationsAdaptDiscreteAll(Dataset):
    def __init__(self, task_ids, df, max_examples=2):
        self.task_ids = task_ids
        self.data_by_task_id = {
            task_id: df.query(f'task_id == "{task_id}"').to_dict(orient='records')
            for task_id in task_ids
        }
        self.max_examples = max_examples
        
    def __len__(self):
        return len(self.task_ids)

    def __getitem__(self, idx):
        task_id = self.task_ids[idx]
        task_data = self.data_by_task_id[task_id]
        
        # Select all non-final examples for the task
        examples = [d for d in task_data if not d['final_task']]
        random.shuffle(examples)  # Shuffle examples
        examples = examples[:self.max_examples]  # Limit to max_examples


        # Get the final task example
        final_task = [d for d in task_data if d['final_task']][0]

        # Apply transformations
        transformation_1 = sample_transformations()
        transformation_2 = lambda x: placeholder(transformation_1(x))
        transformation_32 = lambda x: torch.tensor(transformation_2(x).flatten(), dtype=torch.long)

        # Process all examples
        data_item_input = [transformation_32(ex['input']) for ex in examples]
        data_item_output = [transformation_32(ex['output']) for ex in examples]
        # Process final task
        data_item_input_d = [transformation_32(final_task['input'])]
        data_item_output_d = [transformation_32(final_task['output'])]

        return [
            torch.stack(data_item_input), torch.stack(data_item_output), 
            torch.stack(data_item_input_d), torch.stack(data_item_output_d)
        ]
    

class CustomDatasetTestTransformationsDiscreteWithoutTaskAll(Dataset):
    def __init__(self, task_ids, df, max_examples=2):
        self.task_ids = task_ids
        self.data_by_task_id = {
            task_id: df.query(f'task_id == "{task_id}"').to_dict(orient='records')
            for task_id in task_ids
        }
        self.max_examples = max_examples
        
    def __len__(self):
        return len(self.task_ids)

    def __getitem__(self, idx):
        task_id = self.task_ids[idx]
        task_data = self.data_by_task_id[task_id]
        
        examples = [d for d in task_data if not d['final_task']]
        random.shuffle(examples) 
        final_task = examples.pop()
        examples = examples[:self.max_examples] 

        # Apply transformations
        transformation_1 = sample_transformations()
        transformation_2 = lambda x: placeholder(transformation_1(x))
        transformation_32 = lambda x: torch.tensor(transformation_2(x).flatten(), dtype=torch.long)

        # Process all examples
        data_item_input = [transformation_32(ex['input']) for ex in examples]
        data_item_output = [transformation_32(ex['output']) for ex in examples]
        # Process final task
        data_item_input_d = [transformation_32(final_task['input'])]
        data_item_output_d = [transformation_32(final_task['output'])]

        return [
            torch.stack(data_item_input), torch.stack(data_item_output), 
            torch.stack(data_item_input_d), torch.stack(data_item_output_d)
        ]

#### Models

In [ ]:
from x_transformers import TransformerWrapper, Encoder, Decoder, PrefixDecoder
from x_transformers.autoregressive_wrapper import AutoregressiveWrapper

In [ ]:
class TwoDimPositionalEmbedding(nn.Module):
    def __init__(self, height=32, width=32, dim=32):
        super().__init__()
        self.height = height
        self.width = width
        self.dim = dim

        # Separate embeddings for height and width
        self.row_embed = nn.Embedding(height, dim)
        self.col_embed = nn.Embedding(width, dim)

        # Initialize weights (if needed)
        #nn.init.normal_(self.row_embed.weight, std=0.02)
        #nn.init.normal_(self.col_embed.weight, std=0.02)

    def forward(self, batch_size):
        # Generate grid of row and column indices
        row_positions = torch.arange(self.height, device=self.row_embed.weight.device).unsqueeze(1).expand(self.height, self.width).flatten()
        col_positions = torch.arange(self.width, device=self.row_embed.weight.device).unsqueeze(0).expand(self.height, self.width).flatten()

        # Get embeddings
        row_pos_emb = self.row_embed(row_positions)  # Shape: (height * width, dim)
        col_pos_emb = self.col_embed(col_positions)  # Shape: (height * width, dim)

        # Combine the two positional embeddings
        pos_emb = row_pos_emb + col_pos_emb  # Shape: (height * width, dim)

        # Expand to batch size
        pos_emb = pos_emb.unsqueeze(0).expand(batch_size, -1, -1)  # Shape: (batch_size, height * width, dim)

        return pos_emb

In [ ]:
class EncoderDecoder2dPosModel(nn.Module):
    def __init__(self, num_tokens=NUMBER_OF_COLORS, max_seq_len=MAX_SEQUENCE_SIZE, dim=64, depth=4, heads=8, 
                 pos_emb_2d_encoder=True, pos_emb_2d_decoder=True):
        super().__init__()

        self.dim = dim
        self.depth = depth
        self.heads = heads
        self.pos_emb_2d_encoder = pos_emb_2d_encoder
        self.pos_emb_2d_decoder = pos_emb_2d_decoder

        if pos_emb_2d_encoder or pos_emb_2d_decoder:
            self.pos_enc = TwoDimPositionalEmbedding(height=MAX_IMAGE_SIZE, width=MAX_IMAGE_SIZE, dim=dim)

        self.encoder_model = TransformerWrapper(
            num_tokens = num_tokens,
            max_seq_len = max_seq_len,
            attn_layers = Encoder(
                dim = dim,
                depth = depth,
                heads = heads,
                attn_flash = True
            )
        )
        
        self.decoder = TransformerWrapper(
            num_tokens = num_tokens,
            max_seq_len = max_seq_len,
            attn_layers = Decoder(
                dim = dim,
                depth = depth,
                heads = heads,
                cross_attend = True,
                attn_flash = True
            )
        )

        # If your model is wrapped in DistributedDataParallel (DDP),
        # access the underlying model using model.module
        model_to_freeze = self.module if hasattr(self, "module") else self

        # List of parameter names from the underlying model (without the "module." prefix)
        freeze_params = {
            "encoder_model.pos_emb.emb.weight",
            "encoder_model.to_logits.weight",
            "decoder.pos_emb.emb.weight"
        }

        # Iterate over the underlying model's parameters
        for name, param in model_to_freeze.named_parameters():
            if name in freeze_params:
                param.requires_grad = False
                print(f"Freezing parameter: {name}")

    def forward(self, get_predictions, input_examples, output_examples, input_task, output_task, device):
        loss, _, _, _, _ = get_predictions(self, input_examples, output_examples, input_task, output_task, device=device)
        return loss

    def encode(self, x):
        """
        Encode the input sequence into embeddings.
        
        Parameters:
            x (Tensor): Input tokens of shape [batch_size, sequence_length].
            mask (Tensor, optional): A mask to be applied to the attention mechanism.
        
        Returns:
            Tensor: Embeddings of shape [batch_size, sequence_length, embedding_dim].
        """
        # The TransformerWrapper returns the output embeddings if `return_embeddings=True`
        mask = torch.ones_like(x).bool()

        if self.pos_emb_2d_encoder:
            pos_emb = self.pos_enc(x.shape[0])
            embeddings = self.encoder_model(x, mask=mask, return_embeddings=True, pos = pos_emb)
        else:
            embeddings = self.encoder_model(x, mask=mask, return_embeddings=True)

        return embeddings

    def decode(self, x, context_embeddings, do_prep_target_seq=True):
        """
        Decode embeddings back into a distribution over tokens.
        
        Parameters:
            embeddings (Tensor): Embeddings from the encoder of shape [batch_size, sequence_length, embedding_dim].
            
        Returns:
            Tensor: Logits over tokens of shape [batch_size, sequence_length, num_tokens].
        """
        if self.pos_emb_2d_decoder:
            pos_emb = self.pos_enc(x.shape[0])
            pos_emb = pos_emb[:, :x.shape[1], :] # generate seauence of the same length as x, 1,2,3,...
            logits = self.decoder(prep_target_seq(x) if do_prep_target_seq else x, context=context_embeddings, pos=pos_emb)
        else:
            logits = self.decoder(prep_target_seq(x) if do_prep_target_seq else x, context=context_embeddings)
        return logits
    
    @torch.no_grad()
    def generate(self, context_embeddings, temperature=0):
        """
        Autoregressively generate a sequence given context embeddings.
        
        Parameters:
            context_embeddings (Tensor): Encoder outputs of shape [batch_size, seq_len, dim].
            temperature (float): Sampling temperature. 0 → greedy (argmax).
            max_len (int): Maximum number of tokens to generate.
            start_token (int): Token ID to start decoding with.
        
        Returns:
            Tensor: Generated sequence of shape [batch_size, max_len].
        """
        device = context_embeddings.device
        bs = context_embeddings.size(0)
        
        # Start with a tensor containing only the start token
        generated = torch.full((bs, 1), SOS_TOKEN, dtype=torch.long, device=device)
        
        for _ in range(MAX_SEQUENCE_SIZE - 1):
            logits = self.decode(
                generated,
                context_embeddings=context_embeddings,
                do_prep_target_seq=False
            )
            
            # Take the last predicted token logits
            next_token_logits = logits[:, -1, :]
            
            if temperature == 0:
                # Greedy decoding
                next_token = torch.argmax(next_token_logits, dim=-1, keepdim=True)
            else:
                # Sampling decoding
                probs = torch.softmax(next_token_logits, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)
            
            # Append predicted token
            generated = torch.cat([generated, next_token], dim=1)
        
        return generated

#### Get Predictions and Losses

In [ ]:
import torch
import torch.nn.functional as F

# ------------------------------
# Helper: extract off-diagonal elements
# ------------------------------
def off_diagonal(x):
    n, m = x.shape
    assert n == m
    return x.flatten()[:-1].view(n - 1, n + 1)[:, 1:].flatten()


def latent_space_stats(z1, z2, eps=1e-4):
    n, d = z1.shape

    # 1️⃣ Invariance (MSE)
    sim_loss = F.mse_loss(z1, z2)

    # 2️⃣ Variance
    std_z1 = torch.sqrt(z1.var(dim=0) + eps)
    std_z2 = torch.sqrt(z2.var(dim=0) + eps)
    std_loss = (std_z1.mean() + std_z2.mean()) / 2

    # 3️⃣ Covariance (decorrelation)
    z1 = z1 - z1.mean(dim=0)
    z2 = z2 - z2.mean(dim=0)
    cov_z1 = (z1.T @ z1) / (n - 1)
    cov_z2 = (z2.T @ z2) / (n - 1)
    cov_loss = (off_diagonal(cov_z1).pow(2).mean() +
                off_diagonal(cov_z2).pow(2).mean())

    # ==========================
    # 🔹 Extra Metrics
    # ==========================

    # Combined embeddings (for global norm stats)
    z_all = torch.cat([z1, z2], dim=0)
    norms = z_all.norm(dim=1)
    mean_norm = norms.mean().item()
    std_norm = norms.std().item()

    # Cosine similarity matrix (between z1 and z2)
    cos_matrix = F.cosine_similarity(
        z1.unsqueeze(1), z2.unsqueeze(0), dim=-1
    )  # [n, n]
    mse_matrix = ((z1.unsqueeze(1) - z2.unsqueeze(0)) ** 2).mean(dim=-1)

    # Extract diagonal (pair) and off-diagonal (non-pair)
    mask = ~torch.eye(n, dtype=torch.bool, device=z1.device)
    mean_cos_sim_pairs = cos_matrix.diag().mean().item()
    mean_cos_sim_non_pairs = cos_matrix[mask].mean().item()
    mse_pairs = mse_matrix.diag().mean().item()
    mse_non_pairs = mse_matrix[mask].mean().item()

    collapse_ratio_z1 = (std_z1 < 1e-3).float().mean().item()
    collapse_ratio_z2 = (std_z2 < 1e-3).float().mean().item()
    collapse_ratio = (collapse_ratio_z1 + collapse_ratio_z2) / 2.0

    metrics = {
        "sim_loss": sim_loss.item(),
        "std_loss": std_loss.item(),
        "cov_loss": cov_loss.item(),
        "mean_norm": mean_norm,
        "std_norm": std_norm,
        "mean_cos_sim_pairs": mean_cos_sim_pairs,
        "mean_cos_sim_non_pairs": mean_cos_sim_non_pairs,
        "mse_pairs": mse_pairs,
        "mse_non_pairs": mse_non_pairs,
        "collapse_ratio": collapse_ratio,
    }

    return metrics

In [ ]:
def get_predictions_simple(model, input_examples, output_examples, input_task, output_task, is_train=True, device=device):
    global config

    n_examples = input_examples.shape[1]
    em_dim = model.dim
    bs, n, h, w = input_examples.shape[0], n_examples, MAX_SEQUENCE_SIZE, model.dim
    
    input_examples = input_examples.to(device)
    output_examples = output_examples.to(device)
    input_task = input_task.to(device)
    output_task = output_task.to(device)

    input_examples = input_examples.view(-1, MAX_SEQUENCE_SIZE)
    input_examples_emb = model.encode(input_examples)
    input_examples_emb = input_examples_emb.view(-1, n_examples, MAX_SEQUENCE_SIZE, input_examples_emb.shape[-1])

    output_examples = output_examples.view(-1, MAX_SEQUENCE_SIZE)
    output_examples_emb = model.encode(output_examples)
    output_examples_emb = output_examples_emb.view(-1, n_examples, MAX_SEQUENCE_SIZE, output_examples_emb.shape[-1])

    input_task = input_task.view(-1, MAX_SEQUENCE_SIZE)
    input_task_emb = model.encode(input_task)
    input_task_emb = input_task_emb.view(-1, 1, MAX_SEQUENCE_SIZE, input_task_emb.shape[-1])

    output_task = output_task.view(-1, MAX_SEQUENCE_SIZE)
    output_task_emb = model.encode(output_task)
    output_task_emb = output_task_emb.view(-1, 1, MAX_SEQUENCE_SIZE, output_task_emb.shape[-1])

    # ----------------------------
    if config["C_MEAN_1_2"]:
        c = (output_examples_emb - input_examples_emb).mean(dim=(1, 2), keepdim=True)
    else:
        c = (output_examples_emb - input_examples_emb).mean(dim=(1), keepdim=True)
    #c_norm = torch.mean(c.pow(2)).item()
    c_norm = c.norm(dim=-1).mean().item()
    
    # calculate embedding losses
    input_examples_emb_est = output_examples_emb - c
    output_examples_emb_est = input_examples_emb + c
    input_task_emb_est = output_task_emb - c
    output_task_emb_est = input_task_emb + c

    c = c.mean(dim=(2), keepdim=True) # need for analises

    # ----------------------------
    # # Find transformation as matrix X such that A*X = B
    # # Stack examples per batch: combine (n, h) -> (n*h)
    # A_stacked = input_examples_emb.reshape(bs, n * h, w)  # [bs, n*h, w]
    # B_stacked = output_examples_emb.reshape(bs, n * h, w)  # [bs, n*h, w]

    # ones = torch.ones(bs, n * h, 1, device=A_stacked.device)
    
    # A_aug = torch.cat([A_stacked, ones], dim=-1)           # [bs, n*h, w+1]
    # A_pinv = torch.linalg.pinv(A_aug)                      # [bs, w+1, n*h]
    # X_affine = A_pinv @ B_stacked                          # [bs, w+1, w]
    # # Split into linear and bias parts
    # A_mat = X_affine[:, :w, :]   # [bs, w, w]
    # c_vec = X_affine[:, -1:, :]  # [bs, 1, w]
    
    # # A_inv = torch.linalg.pinv(A_mat)  # [bs, w, w]
    # # c_inv = -c_vec @ A_inv            # [bs, 1, w]
    # B_aug = torch.cat([B_stacked, ones], dim=-1)   # [bs, n*h, w+1]
    # B_pinv = torch.linalg.pinv(B_aug)              # [bs, w+1, n*h]
    # X_affine_inv = B_pinv @ A_stacked              # [bs, w+1, w]
    # A_inv = X_affine_inv[:, :w, :]                 # [bs, w, w]
    # c_inv = X_affine_inv[:, -1:, :]                # [bs, 1, w]

    # # calculate embedding losses
    # input_examples_emb_est = output_examples_emb @ A_inv.unsqueeze(1) + c_inv.unsqueeze(1)
    # output_examples_emb_est = input_examples_emb @ A_mat.unsqueeze(1) + c_vec.unsqueeze(1)
    # input_task_emb_est = output_task_emb @ A_inv.unsqueeze(1) + c_inv.unsqueeze(1)
    # output_task_emb_est = input_task_emb @ A_mat.unsqueeze(1) + c_vec.unsqueeze(1)

    # c_norm = A_mat.abs().mean() + 0.1 * c_vec.abs().mean()
    # ----------------------------

    if config["TRIPLET_LOSS"]:
        triplet_loss = nn.TripletMarginWithDistanceLoss(
            distance_function=nn.PairwiseDistance(), margin=1.0
        )

        input_examples_anchor = input_examples_emb_est.view(-1, MAX_SEQUENCE_SIZE, model.dim).mean(1)
        input_examples_positive = input_examples_emb.view(-1, MAX_SEQUENCE_SIZE, model.dim).mean(1)
        input_examples_negative = output_examples_emb.view(-1, MAX_SEQUENCE_SIZE, model.dim).mean(1)
        input_examples_triplet_loss = triplet_loss(input_examples_anchor, input_examples_positive, input_examples_negative)

        output_examples_anchor = output_examples_emb_est.view(-1, MAX_SEQUENCE_SIZE, model.dim).mean(1)
        output_examples_positive = output_examples_emb.view(-1, MAX_SEQUENCE_SIZE, model.dim).mean(1)
        output_examples_negative = input_examples_emb.view(-1, MAX_SEQUENCE_SIZE, model.dim).mean(1)
        output_examples_triplet_loss = triplet_loss(output_examples_anchor, output_examples_positive, output_examples_negative)

        input_task_anchor = input_task_emb_est.view(-1, MAX_SEQUENCE_SIZE, model.dim).mean(1)
        input_task_positive = input_task_emb.view(-1, MAX_SEQUENCE_SIZE, model.dim).mean(1)
        input_task_negative = output_task_emb.view(-1, MAX_SEQUENCE_SIZE, model.dim).mean(1)
        input_task_triplet_loss = triplet_loss(input_task_anchor, input_task_positive, input_task_negative)

        output_task_anchor = output_task_emb_est.view(-1, MAX_SEQUENCE_SIZE, model.dim).mean(1)
        output_task_positive = output_task_emb.view(-1, MAX_SEQUENCE_SIZE, model.dim).mean(1)
        output_task_negative = input_task_emb.view(-1, MAX_SEQUENCE_SIZE, model.dim).mean(1)
        output_task_triplet_loss = triplet_loss(output_task_anchor, output_task_positive, output_task_negative)

        total_triplet_loss = input_examples_triplet_loss + output_examples_triplet_loss + input_task_triplet_loss + output_task_triplet_loss
    else:
        total_triplet_loss = torch.tensor(0.0)

    # calculate reconstruction losses for embeddings
    # context = torch.no_grad() if not config["EMB_LOSS"] else torch.enable_grad()
    # with context:
    if config["EMB_LOSS"]:
        loss_reconstracted_emb_1 = F.mse_loss(input_examples_emb_est.view(-1, em_dim), input_examples_emb.view(-1, em_dim))
        loss_reconstracted_emb_2 = F.mse_loss(output_examples_emb_est.view(-1, em_dim), output_examples_emb.view(-1, em_dim))
        loss_reconstracted_emb_3 = F.mse_loss(input_task_emb_est.view(-1, em_dim), input_task_emb.view(-1, em_dim))
        loss_reconstracted_emb_4 = F.mse_loss(output_task_emb_est.view(-1, em_dim), output_task_emb.view(-1, em_dim))
        loss_reconstracted_emb = loss_reconstracted_emb_1 + loss_reconstracted_emb_2 + loss_reconstracted_emb_3 + loss_reconstracted_emb_4
    else:
        loss_reconstracted_emb = torch.tensor(0.0)
        loss_reconstracted_emb_1 = torch.tensor(0.0)
        loss_reconstracted_emb_2 = torch.tensor(0.0)
        loss_reconstracted_emb_3 = torch.tensor(0.0)
        loss_reconstracted_emb_4 = torch.tensor(0.0)


    # calculate reconstruction losses for images
    input_examples_est_reconstracted = model.decode(input_examples, input_examples_emb_est.view(-1, MAX_SEQUENCE_SIZE, em_dim))
    output_examples_est_reconstracted = model.decode(output_examples, output_examples_emb_est.view(-1, MAX_SEQUENCE_SIZE, em_dim))
    input_task_est_reconstracted = model.decode(input_task, input_task_emb_est.view(-1, MAX_SEQUENCE_SIZE, em_dim))
    output_task_est_reconstracted = model.decode(output_task, output_task_emb_est.view(-1, MAX_SEQUENCE_SIZE, em_dim))

    with torch.no_grad():
        loss_reconstracted_img_1 = F.cross_entropy(input_examples_est_reconstracted.view(-1, NUMBER_OF_COLORS), input_examples.view(-1))
    loss_reconstracted_img_2 = F.cross_entropy(output_examples_est_reconstracted.view(-1, NUMBER_OF_COLORS), output_examples.view(-1))
    with torch.no_grad():
        loss_reconstracted_img_3 = F.cross_entropy(input_task_est_reconstracted.view(-1, NUMBER_OF_COLORS), input_task.view(-1))
    loss_reconstracted_img_4 = F.cross_entropy(output_task_est_reconstracted.view(-1, NUMBER_OF_COLORS), output_task.view(-1))
    loss_reconstracted_img = loss_reconstracted_img_2 + loss_reconstracted_img_4

    # identity loss
    # context = torch.no_grad() if not config["IDENTITY_LOSS"] else torch.enable_grad()
    # with context:
    if config["IDENTITY_LOSS"]:
        input_examples_est_reconstracted_id = model.decode(input_examples, input_examples_emb.view(-1, MAX_SEQUENCE_SIZE, em_dim))
        output_examples_est_reconstracted_id = model.decode(output_examples, output_examples_emb.view(-1, MAX_SEQUENCE_SIZE, em_dim))
        input_task_est_reconstracted_id = model.decode(input_task, input_task_emb.view(-1, MAX_SEQUENCE_SIZE, em_dim))
        output_task_est_reconstracted_id = model.decode(output_task, output_task_emb.view(-1, MAX_SEQUENCE_SIZE, em_dim))
        loss_identity_img_1 = F.cross_entropy(input_examples_est_reconstracted_id.view(-1, NUMBER_OF_COLORS), input_examples.view(-1))
        loss_identity_img_2 = F.cross_entropy(output_examples_est_reconstracted_id.view(-1, NUMBER_OF_COLORS), output_examples.view(-1))
        loss_identity_img_3 = F.cross_entropy(input_task_est_reconstracted_id.view(-1, NUMBER_OF_COLORS), input_task.view(-1))
        loss_identity_img_4 = F.cross_entropy(output_task_est_reconstracted_id.view(-1, NUMBER_OF_COLORS), output_task.view(-1))
        loss_identity_img = loss_identity_img_1 + loss_identity_img_2 + loss_identity_img_3 + loss_identity_img_4
    else:
        loss_identity_img = torch.tensor(0.0)
        loss_identity_img_1 = torch.tensor(0.0)
        loss_identity_img_2 = torch.tensor(0.0)
        loss_identity_img_3 = torch.tensor(0.0)
        loss_identity_img_4 = torch.tensor(0.0)
    # ----------------------------

    loss = loss_reconstracted_img

    if config["IDENTITY_LOSS"]:
        loss += loss_identity_img

    if config["EMB_LOSS"]:
        loss += 0.01 * loss_reconstracted_emb

    if config["TRIPLET_LOSS"]:
        loss += 0.01 * total_triplet_loss

    if config["C_REG"]:
        #loss += 0.1 * torch.mean(torch.abs(c))
        loss += 0.1 * torch.mean(c.pow(2))

    # log info
    if not is_train:
        with torch.no_grad():
            Ein_seq  = input_task_emb
            Eout_seq = output_task_emb

            # Eout_est_seq = Ein_seq @ A_mat.unsqueeze(1) + c_vec.unsqueeze(1).unsqueeze(1)
            # mse_before = ((Ein_seq - Eout_seq) ** 2).mean(dim=-1)
            # mse_after  = ((Eout_est_seq - Eout_seq) ** 2).mean(dim=-1)
            # frac_better = (mse_after < mse_before).float().mean().item()

            Eout_est_seq = Ein_seq + c
            #Eout_est_seq = Ein_seq @ A_mat.unsqueeze(1) + c_vec.view(bs, 1, 1, w)

            mse_before = ((Ein_seq - Eout_seq) ** 2).mean(dim=-1)
            mse_after  = ((Eout_est_seq - Eout_seq) ** 2).mean(dim=-1)
            mean_relative_improvement = (mse_before.mean() - mse_after.mean()) / mse_before.mean()

            cos_before = F.cosine_similarity(Ein_seq, Eout_seq, dim=-1)
            cos_after  = F.cosine_similarity(Eout_est_seq, Eout_seq, dim=-1)
            cos_gain = (cos_after - cos_before).mean()

        with torch.no_grad():
            estimated_c = c.view(-1, em_dim)

            target_c = (output_task_emb - input_task_emb).mean(dim=(2)).view(-1, em_dim)
            mse_c_target = ((estimated_c - target_c) ** 2).mean().item()
            cos_c_target = F.cosine_similarity(estimated_c.squeeze(1), target_c.squeeze(1), dim=-1).mean().item()

        loss_log = {
            'total_loss': loss.item(),

            'loss_reconstracted_img_1': loss_reconstracted_img_1.item(),
            'loss_reconstracted_img_2': loss_reconstracted_img_2.item(),
            'loss_reconstracted_img_3': loss_reconstracted_img_3.item(),
            'loss_reconstracted_img_4': loss_reconstracted_img_4.item(),
            'loss_reconstracted_img': loss_reconstracted_img.item(),
            
            'loss_reconstracted_emb': loss_reconstracted_emb.item(),
            'loss_reconstracted_emb_1': loss_reconstracted_emb_1.item(),
            'loss_reconstracted_emb_2': loss_reconstracted_emb_2.item(),
            'loss_reconstracted_emb_3': loss_reconstracted_emb_3.item(),
            'loss_reconstracted_emb_4': loss_reconstracted_emb_4.item(),

            'total_triplet_loss': total_triplet_loss.item(),
            'loss_identity': loss_identity_img.item(),
            'c_norm': c_norm,

            'mse_c_target': mse_c_target,
            'cos_c_target': cos_c_target,

            'mean_relative_improvement': mean_relative_improvement.item(),
            'cos_gain': cos_gain.item(),
        }

        if n_examples >=2:
            with torch.no_grad():
                c_log = (output_examples_emb - input_examples_emb).mean(dim=(2))
                z1 = c_log[:, 0, :]
                z2 = c_log[:, 1, :]
                latent_space_metrics = latent_space_stats(z1, z2)

            with torch.no_grad():
                # example_c = (output_examples_emb[:,0,:,:] - input_examples_emb[:,0,:,:]).mean(dim=(1)).view(-1, em_dim)
                # mse_c_example = ((estimated_c - example_c) ** 2).mean().item()
                # cos_c_example = F.cosine_similarity(estimated_c.squeeze(1), example_c.squeeze(1), dim=-1).mean().item()

                # Compute per-example c vectors for all i
                example_c_all = (output_examples_emb - input_examples_emb).mean(dim=2)  # [bs, n, em_dim]
                # Option 1: Compare each example_c_all[:, i, :] with same estimated_c (broadcast)
                mse_c_example_all = ((estimated_c.unsqueeze(1) - example_c_all) ** 2).mean().item()
                cos_c_example_all = F.cosine_similarity(estimated_c.unsqueeze(1), example_c_all, dim=-1).mean().item()
                mse_c_example, cos_c_example = mse_c_example_all, cos_c_example_all

                mean_var_per_vector = estimated_c.std(dim=1, unbiased=False).mean().item()
                mean_var_per_batch = estimated_c.std(dim=0, unbiased=False) .mean().item()


            # with torch.no_grad():
            #     A_stacked_1 = input_examples_emb[:,0,:,:]
            #     B_stacked_1 = output_examples_emb[:,0,:,:]
            #     ones = torch.ones(bs, h, 1, device=A_stacked_1.device)
            #     A_aug_1 = torch.cat([A_stacked_1, ones], dim=-1) 
            #     A_pinv_1= torch.linalg.pinv(A_aug_1)
            #     X_affine_1 = A_pinv_1 @ B_stacked_1

            #     A_stacked_2 = input_examples_emb[:,1,:,:]
            #     B_stacked_2 = output_examples_emb[:,1,:,:]
            #     ones = torch.ones(bs, h, 1, device=A_stacked_2.device)
            #     A_aug_2 = torch.cat([A_stacked_2, ones], dim=-1)
            #     A_pinv_2= torch.linalg.pinv(A_aug_2)
            #     X_affine_2 = A_pinv_2 @ B_stacked_2

            #     # Take first and second examples to compare
            #     z1 = X_affine_1.view(bs, -1)
            #     z2 = X_affine_2.view(bs, -1)

            #     latent_space_metrics = latent_space_stats(z1, z2)


            loss_log_2 = {
                'mse_c_example': mse_c_example,
                'cos_c_example': cos_c_example,

                'mean_var_per_vector': mean_var_per_vector,
                'mean_var_per_batch': mean_var_per_batch,
            }

            loss_log.update(loss_log_2)
            loss_log.update(latent_space_metrics)

    else:
        loss_log = {}
    

    emb_data = (input_examples_emb, output_examples_emb, input_task_emb, output_task_emb)
    emb_data_est = (input_examples_emb_est, output_examples_emb_est, input_task_emb_est, output_task_emb_est)
    decoded_data = (input_examples_est_reconstracted, output_examples_est_reconstracted, input_task_est_reconstracted, output_task_est_reconstracted)

    return loss, emb_data, emb_data_est, decoded_data, loss_log

In [ ]:
def run_eval(model, dataloader, device=device, n_generate_tests=0):
    total_loss = 0.0
    total_loss_reconstracted_emb = 0.0
    total_loss_reconstracted_img = 0.0

    total_loss_reconstracted_img_1 = 0.0
    total_loss_reconstracted_img_2 = 0.0
    total_loss_reconstracted_img_3 = 0.0
    total_loss_reconstracted_img_4 = 0.0

    loss_reconstracted_emb_1 = 0.0
    loss_reconstracted_emb_2 = 0.0
    loss_reconstracted_emb_3 = 0.0
    loss_reconstracted_emb_4 = 0.0

    all_correct_img_1 = 0
    all_correct_img_2 = 0
    all_correct_img_3 = 0
    all_correct_img_4 = 0

    proportion_correct_img_1 = 0
    proportion_correct_img_2 = 0
    proportion_correct_img_3 = 0
    proportion_correct_img_4 = 0

    total_triplet_loss = 0

    mse_c_target = 0
    cos_c_target = 0
    mse_c_example = 0
    cos_c_example = 0

    mean_var_per_vector = 0
    mean_var_per_batch = 0

    sim_loss = 0
    std_loss = 0
    cov_loss = 0
    mean_norm = 0
    std_norm = 0
    mean_cos_sim_pairs = 0
    mean_cos_sim_non_pairs = 0
    mse_pairs = 0
    mse_non_pairs = 0
    collapse_ratio = 0

    loss_identity = 0
    mean_relative_improvement = 0
    cos_gain = 0
    c_norm = 0

    output_task_emb_better_total = 0

    K=1

    with torch.no_grad():
        model.eval()

        for batch_idx, (input_examples, output_examples, input_task, output_task) in enumerate(dataloader):
            loss, emb_data, emb_data_est, (input_examples_est_reconstracted, output_examples_est_reconstracted, input_task_est_reconstracted, output_task_est_reconstracted), loss_log = get_predictions_simple(model, input_examples, output_examples, input_task, output_task, False)
            input_examples_emb, output_examples_emb, input_task_emb, output_task_emb = emb_data
            input_examples_emb_est, output_examples_emb_est, input_task_emb_est, output_task_emb_est = emb_data_est

        # for batch_idx, (input_examples, output_examples, input_task, output_task, mask) in enumerate(dataloader):
        #     loss, _, (input_examples_est_reconstracted, output_examples_est_reconstracted, input_task_est_reconstracted, output_task_est_reconstracted), loss_log = get_predictions_simple_mask(model, input_examples, output_examples, input_task, output_task, mask, device)

            total_loss += loss.item()
            total_loss_reconstracted_emb += loss_log.get('loss_reconstracted_emb', 0)
            total_loss_reconstracted_img += loss_log.get('loss_reconstracted_img', 0)
            total_loss_reconstracted_img_1 += loss_log.get('loss_reconstracted_img_1', 0)
            total_loss_reconstracted_img_2 += loss_log.get('loss_reconstracted_img_2', 0)
            total_loss_reconstracted_img_3 += loss_log.get('loss_reconstracted_img_3', 0)
            total_loss_reconstracted_img_4 += loss_log.get('loss_reconstracted_img_4', 0)
            loss_reconstracted_emb_1 += loss_log.get('loss_reconstracted_emb_1', 0)
            loss_reconstracted_emb_2 += loss_log.get('loss_reconstracted_emb_2', 0)
            loss_reconstracted_emb_3 += loss_log.get('loss_reconstracted_emb_3', 0)
            loss_reconstracted_emb_4 += loss_log.get('loss_reconstracted_emb_4', 0)

            # ---- TOP-K ACCURACY CALCULATION ----
            input_examples = input_examples.view(-1, MAX_SEQUENCE_SIZE).to(device)
            output_examples = output_examples.view(-1, MAX_SEQUENCE_SIZE).to(device)
            input_task = input_task.view(-1, MAX_SEQUENCE_SIZE).to(device)
            output_task = output_task.view(-1, MAX_SEQUENCE_SIZE).to(device)

            # For each set of predicted logits, find top-k predictions
            # input2_est_reconstracted: [batch_size, MAX_SEQUENCE_SIZE, NUMBER_OF_COLORS]
            topk_input2 = torch.topk(input_examples_est_reconstracted, k=K, dim=-1).indices  # shape: [batch_size, MAX_SEQUENCE_SIZE, K]
            topk_output2 = torch.topk(output_examples_est_reconstracted, k=K, dim=-1).indices
            topk_input_21 = torch.topk(input_task_est_reconstracted, k=K, dim=-1).indices
            topk_output_21 = torch.topk(output_task_est_reconstracted, k=K, dim=-1).indices

            # Check for correctness: a pixel is correct if the ground truth is in the top-k predictions
            correct_pixels_img_1 = (topk_input2 == input_examples.unsqueeze(-1)).any(dim=-1)
            correct_pixels_img_2 = (topk_output2 == output_examples.unsqueeze(-1)).any(dim=-1)
            correct_pixels_img_3 = (topk_input_21 == input_task.unsqueeze(-1)).any(dim=-1)
            correct_pixels_img_4 = (topk_output_21 == output_task.unsqueeze(-1)).any(dim=-1)

            # all_correct_img_x: count how many images have all pixels correct
            all_correct_img_1 += torch.sum(torch.all(correct_pixels_img_1, dim=1)).item()
            all_correct_img_2 += torch.sum(torch.all(correct_pixels_img_2, dim=1)).item()
            all_correct_img_3 += torch.sum(torch.all(correct_pixels_img_3, dim=1)).item()
            all_correct_img_4 += torch.sum(torch.all(correct_pixels_img_4, dim=1)).item()

            # proportion_correct_img_x: average proportion of correctly predicted pixels
            proportion_correct_img_1 += correct_pixels_img_1.float().mean().item()
            proportion_correct_img_2 += correct_pixels_img_2.float().mean().item()
            proportion_correct_img_3 += correct_pixels_img_3.float().mean().item()
            proportion_correct_img_4 += correct_pixels_img_4.float().mean().item()

            # perplexity
            ppl_img1 = math.exp(loss_log.get('loss_reconstracted_img_1', 0))
            ppl_img2 = math.exp(loss_log.get('loss_reconstracted_img_2', 0))
            ppl_img3 = math.exp(loss_log.get('loss_reconstracted_img_3', 0))
            ppl_img4 = math.exp(loss_log.get('loss_reconstracted_img_4', 0))

            loss_identity += loss_log.get('loss_identity', 0)
            mean_relative_improvement += loss_log.get('mean_relative_improvement', 0)
            cos_gain += loss_log.get('cos_gain', 0)
            c_norm += loss_log.get('c_norm', 0)

            sim_loss += loss_log.get('sim_loss', 0)
            std_loss += loss_log.get('std_loss', 0)
            cov_loss += loss_log.get('cov_loss', 0)
            mean_norm += loss_log.get('mean_norm', 0)
            std_norm += loss_log.get('std_norm', 0)
            mean_cos_sim_pairs += loss_log.get('mean_cos_sim_pairs', 0)
            mean_cos_sim_non_pairs += loss_log.get('mean_cos_sim_non_pairs', 0)
            mse_pairs += loss_log.get('mse_pairs', 0)
            mse_non_pairs += loss_log.get('mse_non_pairs', 0)
            collapse_ratio += loss_log.get('collapse_ratio', 0)

            total_triplet_loss += loss_log.get('total_triplet_loss', 0)

            mse_c_target += loss_log.get('mse_c_target', 0)
            cos_c_target += loss_log.get('cos_c_target', 0)
            mse_c_example += loss_log.get('mse_c_example', 0)
            cos_c_example += loss_log.get('cos_c_example', 0)

            mean_var_per_vector += loss_log.get('mean_var_per_vector', 0)
            mean_var_per_batch += loss_log.get('mean_var_per_batch', 0)

            # --- inside loop, after emb_data unpacking ---
            with torch.no_grad():
                d_true = torch.norm(output_task_emb_est.view(output_task_emb.size(0), -1) -
                                    output_task_emb.view(output_task_emb.size(0), -1), dim=-1)
                d_input = torch.norm(output_task_emb_est.view(output_task_emb.size(0), -1) -
                                    input_task_emb.view(input_task_emb.size(0), -1), dim=-1)
                better_mask = (d_true < d_input).float()
                output_task_emb_better_total += better_mask.mean().item()


    total_images = len(dataloader)

    avg_proportion_correct_img_1 = proportion_correct_img_1 / total_images
    avg_proportion_correct_img_2 = proportion_correct_img_2 / total_images
    avg_proportion_correct_img_3 = proportion_correct_img_3 / total_images
    avg_proportion_correct_img_4 = proportion_correct_img_4 / total_images

    total_loss, total_loss_reconstracted_emb, total_loss_reconstracted_img = total_loss / total_images, total_loss_reconstracted_emb / total_images, total_loss_reconstracted_img / total_images
    total_loss_reconstracted_img_1, total_loss_reconstracted_img_2, total_loss_reconstracted_img_3, total_loss_reconstracted_img_4 = total_loss_reconstracted_img_1 / total_images, total_loss_reconstracted_img_2 / total_images, total_loss_reconstracted_img_3 / total_images, total_loss_reconstracted_img_4 / total_images
    loss_reconstracted_emb_1, loss_reconstracted_emb_2, loss_reconstracted_emb_3, loss_reconstracted_emb_4 = loss_reconstracted_emb_1 / total_images, loss_reconstracted_emb_2 / total_images, loss_reconstracted_emb_3 / total_images, loss_reconstracted_emb_4 / total_images

    loss_identity /= total_images
    
    mean_relative_improvement /= total_images
    cos_gain /= total_images
    c_norm /= total_images

    sim_loss /= total_images
    std_loss /= total_images
    cov_loss /= total_images
    mean_norm /= total_images
    std_norm /= total_images
    mean_cos_sim_pairs /= total_images
    mean_cos_sim_non_pairs /= total_images
    mse_pairs /= total_images
    mse_non_pairs /= total_images
    collapse_ratio /= total_images

    mse_c_target /= total_images
    cos_c_target /= total_images
    mse_c_example /= total_images
    cos_c_example /= total_images

    total_triplet_loss /= total_images

    mean_var_per_vector /= total_images
    mean_var_per_batch /= total_images

    output_task_emb_better_total /= total_images

    # print('All correct gueses', len(train_loader_transformations.dataset), all_correct_img_1, all_correct_img_2, all_correct_img_3, all_correct_img_4)
    # print('Proportion of correct pixels for each image:', avg_proportion_correct_img_1, avg_proportion_correct_img_2, avg_proportion_correct_img_3, avg_proportion_correct_img_4)
    # print('Total loss', total_loss, total_loss_reconstracted_emb, total_loss_reconstracted_img)

    data = {
        "total_loss": total_loss,
        "total_loss_reconstracted_emb": total_loss_reconstracted_emb,
        "total_loss_reconstracted_img": total_loss_reconstracted_img,
        "total_loss_reconstracted_img_1": total_loss_reconstracted_img_1,
        "total_loss_reconstracted_img_2": total_loss_reconstracted_img_2,
        "total_loss_reconstracted_img_3": total_loss_reconstracted_img_3,
        "total_loss_reconstracted_img_4": total_loss_reconstracted_img_4,
        "loss_reconstracted_emb_1": loss_reconstracted_emb_1,
        "loss_reconstracted_emb_2": loss_reconstracted_emb_2,
        "loss_reconstracted_emb_3": loss_reconstracted_emb_3,
        "loss_reconstracted_emb_4": loss_reconstracted_emb_4,

        "avg_proportion_correct_img_1": avg_proportion_correct_img_1,
        "avg_proportion_correct_img_2": avg_proportion_correct_img_2,
        "avg_proportion_correct_img_3": avg_proportion_correct_img_3,
        "avg_proportion_correct_img_4": avg_proportion_correct_img_4,

        "all_correct_img_1": all_correct_img_1,
        "all_correct_img_2": all_correct_img_2,
        "all_correct_img_3": all_correct_img_3,
        "all_correct_img_4": all_correct_img_4,

        "perplexity_img_1": ppl_img1,
        "perplexity_img_2": ppl_img2,
        "perplexity_img_3": ppl_img3,
        "perplexity_img_4": ppl_img4,

        "loss_identity": loss_identity,
        "mean_relative_improvement": mean_relative_improvement,   
        "cos_gain": cos_gain,
        "c_norm": c_norm,

        "sim_loss": sim_loss,
        "std_loss": std_loss,
        "cov_loss": cov_loss,
        "mean_norm": mean_norm,
        "std_norm": std_norm,
        "mean_cos_sim_pairs": mean_cos_sim_pairs,
        "mean_cos_sim_non_pairs": mean_cos_sim_non_pairs,
        "mse_pairs": mse_pairs,
        "mse_non_pairs": mse_non_pairs,
        "collapse_ratio": collapse_ratio,

        "mse_c_target": mse_c_target,
        "cos_c_target": cos_c_target,
        "mse_c_example": mse_c_example,
        "cos_c_example": cos_c_example,

        "total_triplet_loss": total_triplet_loss,

        "mean_var_per_vector": mean_var_per_vector,
        "mean_var_per_batch": mean_var_per_batch,

        "output_task_emb_better_total": output_task_emb_better_total,
    }

    return data

#### TTT

In [ ]:
def run_fine_tune(model, optimizer, train_loader, test_loader, n_epoch_finetune, grad_accum_steps=1):
    num_warmup_batches = 4  # Set the number of warm-up batches
    def lr_lambda(current_step):
        if current_step < num_warmup_batches:
            return float(current_step) / float(max(1, num_warmup_batches))
        return 1.0
    scheduler = LambdaLR(optimizer, lr_lambda=lr_lambda)

    for epoch in range(n_epoch_finetune):
        model.train()
        total_loss = 0

        for batch_idx, (input_examples, output_examples, input_task, output_task) in enumerate(train_loader):
            with torch.autocast(device_type=str(device), dtype=torch.bfloat16):
                loss, _, _, _, _ = get_predictions_simple(model, input_examples, output_examples, input_task, output_task)
                loss /= grad_accum_steps

            # Accumulating gradients over multiple steps
            loss.backward()  # Retain graph for reuse during accumulation
            
            # Check if it's time to update the weights
            if (batch_idx + 1) % grad_accum_steps == 0:
                # Clip gradients to avoid exploding gradients
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                
                # Update weights
                optimizer.step()
                scheduler.step()
                
                # Zero gradients for the next accumulation step
                optimizer.zero_grad()

            total_loss += loss.item()

        with torch.no_grad():
            total_loss = total_loss / len(train_loader) * grad_accum_steps 
            print(f"Epoch {epoch + 1} completed with total loss: {total_loss}", )
            
            # model.eval()
            # train_data = run_eval(model, train_loader)
            # test_data = run_eval(model, test_loader)
            # print(f"Epoch {epoch + 1} completed with total loss: {total_loss}", 'Train:', train_data['avg_proportion_correct_img_4'], train_data['all_correct_img_4'], train_data['total_loss_reconstracted_img_4'], 'Test:', test_data['avg_proportion_correct_img_4'], test_data['all_correct_img_4'], test_data['total_loss_reconstracted_img_4'])


        #     # # log images
        #     img_train_name, img_test_name = store_test_as_image(model, train_loader, test_loader, epoch=epoch)
        #     wandb.log({"train_images": wandb.Image(img_train_name), "test_images": wandb.Image(img_test_name)})
    

    return model

In [ ]:
def ttt_pipeline(config, model_name, run_id=0, n_tasks=None, use_wandb=True):
    if os.path.exists(model_name):
        print("Model exists!", model_name)
    else:
        print("File does NOT exist!", model_name)
        return
    
    if n_tasks is None:
        n_tasks = 400        # full evaluation
    else:
        n_tasks = int(n_tasks)       # e.g., for fast mode (40)

    debug_task_ids = test_task_ids # train_task_ids
    debug_df = df_test #df_train

    em_dim = 256
    depth = 4
    model = EncoderDecoder2dPosModel(dim=em_dim, depth=depth, heads=8, 
                                     pos_emb_2d_encoder=config['pos_emb_2d_encoder'], 
                                     pos_emb_2d_decoder=config['pos_emb_2d_decoder']).to(device)
    #model = torch.compile(model)

    n_epoch_finetune = 2
    n_train_set_finetune = 256
    n_test_set = 256
    bs_finetune = 8
    bs_test = bs_finetune * 2
    grad_accum_steps = 1
    learning_rate = 3e-4

    if use_wandb:
        wandb.init(project="eval_ttt_erd", 
                name=f"{config['NAME']}_run_{run_id}",
                    group=config["NAME"],
                config={
            "learning_rate": learning_rate,
            "batch_size": bs_finetune*grad_accum_steps,
            "em_dim": em_dim,
            "depth": depth,
            "n_epoch_finetune": n_epoch_finetune,
            "n_train_set_finetune": n_train_set_finetune,
            "model_size": count_parameters(model)
        })

    n_correct_per_task_before = []
    n_correct_per_task_after = []
    tasks_with_zero_solved_rate = []

    #for task_id in [4]:
    for task_id in range(n_tasks):
        # clear cache
        torch.cuda.empty_cache()

        print(f'Progress {task_id} over {n_tasks}:')
        # load model
        model.load_state_dict(torch.load(model_name, weights_only=True)["model"])
        optimizer = AdamW(model.parameters(), lr=learning_rate, betas=(0.9, 0.95))
        
        # train loader
        test_task_ids_single = [debug_task_ids[task_id]] * n_train_set_finetune
        dataset_train = CustomDatasetTestTransformationsDiscreteWithoutTaskAll(test_task_ids_single, debug_df, max_examples=config['N_OF_EXAMPLES'])
        train_adaptation_loader = DataLoader(dataset_train, batch_size=bs_finetune, shuffle=True, drop_last=True)
        # eval loaders
        test_task_ids_single = [debug_task_ids[task_id]] * n_test_set
        dataset_test_eval = CustomDatasetTestTransformationsAdaptDiscreteAll(test_task_ids_single, debug_df, max_examples=config['N_OF_EXAMPLES'])
        eval_adaptation_loader = DataLoader(dataset_test_eval, batch_size=bs_test, shuffle=True, drop_last=True)

        # evaluate before fine-tuning
        with torch.no_grad():
            model.eval()
            train_data = run_eval(model, train_adaptation_loader)
            test_data = run_eval(model, eval_adaptation_loader)
            n_correct_per_task_before.append(test_data['all_correct_img_4'] / n_test_set)
            
            print('Before.', 
                'Train:', 
                'avg prop correct px =', f"{train_data['avg_proportion_correct_img_4']:.4f}", 
                'all correct =', f"{train_data['all_correct_img_4'] / n_train_set_finetune :.4f}", 
                'Test:', 
                'avg prop correct px =', f"{test_data['avg_proportion_correct_img_4']:.4f}", 
                'all correct =', f"{test_data['all_correct_img_4'] / n_test_set:.4f}")

        # fine tune
        model = run_fine_tune(model, optimizer, train_adaptation_loader, eval_adaptation_loader, n_epoch_finetune, grad_accum_steps)

        # max_runs = 10
        # run_count = 0
        # stop_when_n_solved = 200
        # while True:
        #     with torch.no_grad():
        #         model.eval()
        #         train_data = run_eval(model, train_adaptation_loader)
        #         test_data = run_eval(model, eval_adaptation_loader)
        #         print(f'Attempt {run_count + 1}/{max_runs},', train_data['avg_proportion_correct_img_4'], train_data['all_correct_img_4'], test_data['avg_proportion_correct_img_4'], test_data['all_correct_img_4'])

        #     if train_data['all_correct_img_4'] > stop_when_n_solved or run_count >= max_runs:
        #         break

        #     model = run_fine_tune(model, optimizer, train_adaptation_loader, n_epoch_finetune)
        #     run_count += 1

        # evaluate after fine-tuning
        with torch.no_grad():
            model.eval()
            train_data = run_eval(model, train_adaptation_loader)
            test_data = run_eval(model, eval_adaptation_loader)
            n_correct_per_task_after.append(test_data['all_correct_img_4'] / n_test_set)

            print('After.', 
                'Train:', 
                'avg prop correct px =', f"{train_data['avg_proportion_correct_img_4']:.4f}", 
                'all correct =', f"{train_data['all_correct_img_4'] / n_train_set_finetune :.4f}", 
                'Test:', 
                'avg prop correct px =', f"{test_data['avg_proportion_correct_img_4']:.4f}", 
                'all correct =', f"{test_data['all_correct_img_4'] / n_test_set:.4f}")

        if n_correct_per_task_after[-1] == 0:
            tasks_with_zero_solved_rate.append(debug_task_ids[task_id])
        
        
        print(f'Current task. Before = {n_correct_per_task_before[-1]:.4f}, After = {n_correct_per_task_after[-1]:.4f}')  
        print(f'All tasks mean. Before = {np.mean(n_correct_per_task_before):.4f}, After = {np.mean(n_correct_per_task_after):.4f}')
        print(f'All tasks std. Before = {np.std(n_correct_per_task_before):.4f}, After = {np.std(n_correct_per_task_after):.4f}')
        print('Zero solved rate: ', len(tasks_with_zero_solved_rate), tasks_with_zero_solved_rate)
        print()


        if use_wandb:
            # log
            wandb.log({
                'proportion_solved_before': np.mean(n_correct_per_task_before),
                'proportion_solved_after': np.mean(n_correct_per_task_after),
                'proportion_solved_before_var': np.std(n_correct_per_task_before),
                'proportion_solved_after_var': np.std(n_correct_per_task_after),
                'zero_solved_rate': len(tasks_with_zero_solved_rate),
                'proportion_with_solution': ((task_id + 1) - len(tasks_with_zero_solved_rate)) / (task_id + 1),
                'proportion_with_no_solution': len(tasks_with_zero_solved_rate) / (task_id + 1)
            })

    # # log summary
    # wandb.summary['final_proportion_solved_before'] = np.mean(n_correct_per_task_before)
    # wandb.summary['final_proportion_solved_after'] = np.mean(n_correct_per_task_after)
    # wandb.summary['final_proportion_solved_before_var'] = np.std(n_correct_per_task_before)
    # wandb.summary['final_proportion_solved_after_var'] = np.std(n_correct_per_task_after)
    # wandb.summary['final_zero_solved_rate'] = len(tasks_with_zero_solved_rate)
    # wandb.summary['final_proportion_with_solution'] = (400 - len(tasks_with_zero_solved_rate)) / 400

    # wandb.finish()

    # --- final summary ---
    summary = {
        "final_proportion_solved_before": np.mean(n_correct_per_task_before),
        "final_proportion_solved_after": np.mean(n_correct_per_task_after),
        "final_proportion_solved_before_var": np.std(n_correct_per_task_before),
        "final_proportion_solved_after_var": np.std(n_correct_per_task_after),
        "final_zero_solved_rate": len(tasks_with_zero_solved_rate),
        "final_proportion_with_solution": (n_tasks - len(tasks_with_zero_solved_rate)) / n_tasks,
        "final_proportion_with_no_solution": len(tasks_with_zero_solved_rate) / n_tasks
    }

    if use_wandb:
        for k, v in summary.items():
            wandb.summary[f"final_{k}"] = v
        wandb.finish()

    return summary

In [ ]:
# config = {
#     "IDENTITY_LOSS": False,
#     "EMB_LOSS": False,
#     "TRIPLET_LOSS": False,
#     "C_REG": False,

#     "pos_emb_2d_encoder": True, 
#     "pos_emb_2d_decoder": False,

#     "C_MEAN_1_2: True,

#     "N_OF_EXAMPLES": 1,
    
#     "NAME": "erd_base_1_example",
#     "BS": 8
# }


# for run_id in range(1):
#     print(f"TTT pipeline {run_id}")

#     #model_name = f"best_model_{config['NAME']}_run_{run_id}.pt"
#     model_name = "model_erd_epoch_30000.pt"
#     ttt_pipeline(config, model_name, run_id)


# config["NAME"] = "final_erd_epoch_10000"
# model_name = "model_erd_epoch_10000.pt"
# ttt_summary = ttt_pipeline(config, model_name)
# print('TTT Summary:', ttt_summary)

# raise ValueError("Stop here")

#### Multy GPU

In [ ]:
config = {
    "project": "arc_paper_final",
    "run_name": "erd_base",
    "group": "erd_base",

    "IDENTITY_LOSS": False,
    "EMB_LOSS": False,
    "TRIPLET_LOSS": False,
    "C_REG": False,

    "pos_emb_2d_encoder": True, 
    "pos_emb_2d_decoder": False,

    "C_MEAN_1_2": True,

    'N_OF_EXAMPLES': 1,

    "em_dim": 256,
    "depth": 4,
    "heads": 8,
    
    "max_epochs": 100001,
    "save_every": 10000,

    "batch_size": 8,
    "learning_rate": 3e-4 * 4, # update by multiply xN if N gpu, x4 if 4 gpus
    "warmup_epochs": 512,
    "snapshot_path": "model_erd.pt",
}

run_name = f"erd_base__N_EX_{config['N_OF_EXAMPLES']}__2d_encoder_{str(config['pos_emb_2d_encoder'])}__C_MEAN_1_2_{str(config['C_MEAN_1_2'])}"
config["run_name"] = run_name
config["group"] = run_name
config["snapshot_path"] = f"model_{run_name}.pt"
print('Model name:', run_name)

In [ ]:
batch_size_train = config['batch_size']
n_of_samples = config['N_OF_EXAMPLES']

train_eval_loader_transformations = DataLoader(CustomDatasetTestTransformationsAdaptDiscrete(train_task_ids, df_train, n_of_samples=n_of_samples), batch_size=batch_size_train*3)
test_eval_loader_transformations = DataLoader(CustomDatasetTestTransformationsAdaptDiscrete(test_task_ids, df_test, n_of_samples=n_of_samples), batch_size=batch_size_train*3)
train_arc2_eval_loader_transformations = DataLoader(CustomDatasetTestTransformationsAdaptDiscrete(train_task_ids_arc2, df_train_arc2, n_of_samples=n_of_samples), batch_size=batch_size_train*3)
test_arc2_eval_loader_transformations = DataLoader(CustomDatasetTestTransformationsAdaptDiscrete(test_task_ids_arc2, df_test_arc2, n_of_samples=n_of_samples), batch_size=batch_size_train*3)


def run_eval_ddp(model, epoch):
    model.eval()

    with torch.no_grad():
        model.eval()
        train_data = run_eval(model, train_eval_loader_transformations, n_generate_tests=0)
        print('Train ARC1', train_data)
        test_data = run_eval(model, test_eval_loader_transformations, n_generate_tests=0)
        print('Test ARC1', test_data)
        train_arc2_data = run_eval(model, train_arc2_eval_loader_transformations, n_generate_tests=0)
        print('Train ARC2', train_arc2_data)
        test_arc2_data = run_eval(model, test_arc2_eval_loader_transformations, n_generate_tests=0)
        print('Test ARC2', test_arc2_data)

        # Add prefixes to the keys in train and test data
        train_data_prefixed = {f"train_{key}": value for key, value in train_data.items()}
        test_data_prefixed = {f"test_{key}": value for key, value in test_data.items()}
        train_arc2_data_prefixed = {f"train_arc_2_{key}": value for key, value in train_arc2_data.items()}
        test_arc2_data_prefixed = {f"test_arc_2_{key}": value for key, value in test_arc2_data.items()}

    # Log the data with wandb
    wandb.log(train_data_prefixed, step=epoch)
    wandb.log(test_data_prefixed, step=epoch)
    wandb.log(train_arc2_data_prefixed, step=epoch)
    wandb.log(test_arc2_data_prefixed, step=epoch)

    model.train()

In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torch.multiprocessing as mp
from torch.utils.data.distributed import DistributedSampler
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.distributed import init_process_group, destroy_process_group
import os

import copy


def ddp_setup():
    torch.cuda.set_device(int(os.environ["LOCAL_RANK"]))
    init_process_group(backend="nccl")

def ddp_average(value: torch.Tensor):
    avg = value.detach().clone()
    torch.distributed.all_reduce(avg, op=torch.distributed.ReduceOp.SUM)
    avg /= torch.distributed.get_world_size()
    return avg

class Trainer:
    def __init__(
        self,
        model: torch.nn.Module,
        train_data: DataLoader,
        optimizer: torch.optim.Optimizer,
        save_every: int,
        snapshot_path: str,
    ) -> None:
        self.gpu_id = int(os.environ["LOCAL_RANK"])
        self.model = model.to(self.gpu_id)
        self.train_data = train_data
        self.optimizer = optimizer
        self.save_every = save_every
        self.epochs_run = 0
        self.snapshot_path = snapshot_path

        # if os.path.exists(snapshot_path):
        #     print("Loading snapshot")
        #     self._load_snapshot(snapshot_path)

        self.model = DDP(self.model, device_ids=[self.gpu_id])

    def _load_snapshot(self, snapshot_path):
        loc = f"cuda:{self.gpu_id}"
        snapshot = torch.load(snapshot_path, map_location=loc)
        self.model.load_state_dict(snapshot)
        print(f"Resuming training from snapshot at Epoch {self.epochs_run}")

    # def _run_batch(self, input_examples, output_examples, input_task, output_task):
    #     self.optimizer.zero_grad()

    #     with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
    #         loss = get_predictions_simple(self.model, input_examples, output_examples, input_task, output_task, device=self.gpu_id)
    #     #print('Batch Loss:', loss.item())

    #     loss.backward()
    #     # ---- Gradient clipping ----
    #     torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
    #     self.optimizer.step()

    # def _run_epoch(self, epoch):
    #     b_sz = len(next(iter(self.train_data))[0])
    #     current_lr = self.optimizer.param_groups[0]["lr"]

    #     print(f"[GPU{self.gpu_id}] Epoch {epoch} | Batchsize: {b_sz} | Learning rate: {current_lr} | Steps: {len(self.train_data)}")
    #     self.train_data.sampler.set_epoch(epoch)
    #     for input_examples, output_examples, input_task, output_task in self.train_data:
    #         self._run_batch(input_examples, output_examples, input_task, output_task)

    def _run_epoch(self, epoch):
        b_sz = len(next(iter(self.train_data))[0])
        current_lr = self.optimizer.param_groups[0]["lr"]

        print(f"[GPU{self.gpu_id}] Epoch {epoch} | Batchsize: {b_sz} | LR: {current_lr} | Steps: {len(self.train_data)}")

        # Important for DistributedSampler
        self.train_data.sampler.set_epoch(epoch)

        epoch_loss = 0.0
        num_batches = 0

        for input_examples, output_examples, input_task, output_task in self.train_data:
            self.optimizer.zero_grad()

            with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                loss = self.model(get_predictions_simple,
                    input_examples,
                    output_examples,
                    input_task,
                    output_task,
                    device=self.gpu_id
                )

            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            self.optimizer.step()

            epoch_loss += loss.detach()
            num_batches += 1

        # ---- Local loss (this GPU only) ----
        local_epoch_loss = epoch_loss / num_batches

        # ---- Global averaged loss across all GPUs ----
        global_epoch_loss = ddp_average(local_epoch_loss)

        # ---- Log only from GPU 0 ----
        if self.gpu_id == 0:
            wandb.log({
                "total_train_loss": global_epoch_loss.item(),
                "lr": current_lr,
                "epoch": epoch
            }, step=epoch)

        # ---- Print losses (every GPU prints its own local + global) ----
        print(
            f"[GPU{self.gpu_id}] Epoch {epoch} | "
            f"Local loss: {local_epoch_loss.item():.4f} | "
            f"Global loss: {global_epoch_loss.item():.4f}"
        )

    def _save_snapshot(self, epoch):
        # snapshot = {
        #     "epoch": epoch,
        #     "model": self.model.module.state_dict(),
        #     "optimizer": self.optimizer.state_dict()
        # }

        snapshot = self.model.module.state_dict()

        # Example: snapshot.pt → snapshot_epoch_10.pt
        epoch_path = self.snapshot_path.replace(".pt", f"_epoch_{epoch}.pt")

        torch.save(snapshot, epoch_path)
        print(f"Epoch {epoch} | Training snapshot saved at {epoch_path}")

        wandb.save(epoch_path, policy="now")

    def train(self, max_epochs: int, scheduler=None):
        for epoch in range(self.epochs_run, max_epochs):
            self._run_epoch(epoch)

            # step scheduler ONCE per epoch (safe for DDP)
            if scheduler is not None:
                scheduler.step()

            # ---- Evaluation every N epochs ----
            if self.gpu_id == 0 and (epoch % 1000 == 0):
                eval_model = self.model.module
                run_eval_ddp(eval_model, epoch)

            if self.gpu_id == 0 and epoch % self.save_every == 0:
                self._save_snapshot(epoch)


def load_train_objs(config):
    train_set = CustomDatasetTestTransformationsDiscreteReArc2(train_task_ids, df_train, n_of_samples=config['N_OF_EXAMPLES'])
    #train_set = CustomDatasetTestTransformationsAdaptDiscrete(train_task_ids, df_train)

    model = EncoderDecoder2dPosModel(dim=config["em_dim"], depth=config["depth"], heads=config["heads"], 
                                    pos_emb_2d_encoder=config['pos_emb_2d_encoder'], 
                                    pos_emb_2d_decoder=config['pos_emb_2d_decoder']).to(device)

    learning_rate = config["learning_rate"]
    optimizer = AdamW(model.parameters(), lr=learning_rate, betas=(0.9, 0.95))

    # ---- Warmup scheduler ----
    num_warmup_epochs = config["warmup_epochs"]
    def lr_lambda(current_epoch):
        if current_epoch < num_warmup_epochs:
            return float(current_epoch + 1) / float(num_warmup_epochs)
        return 1.0
    scheduler = LambdaLR(optimizer, lr_lambda=lr_lambda)

    return train_set, model, optimizer, scheduler



def prepare_dataloader(dataset: Dataset, batch_size: int):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        pin_memory=True,
        drop_last=True,
        shuffle=False,
        sampler=DistributedSampler(dataset)
    )


def main(config):
    ddp_setup()

    dataset, model, optimizer, scheduler = load_train_objs(config)
    train_data = prepare_dataloader(dataset, config["batch_size"])
    trainer = Trainer(model, train_data, optimizer, config["save_every"], config["snapshot_path"])
    
    # Only GPU0 initializes wandb
    if int(os.environ["LOCAL_RANK"]) == 0:
        # --- create copy ---
        wandb_config = copy.deepcopy(config)
        # --- update only in the copy ---
        wandb_config["batch_size"] = wandb_config["batch_size"] * torch.distributed.get_world_size()
        # Optional: log model size
        wandb_config["model_size"] = count_parameters(model)

        wandb.init(
            project=config["project"],
            name=config["run_name"],
            group=config["group"],
            config=wandb_config
        )
    
    trainer.train(config["max_epochs"], scheduler)

    destroy_process_group()


if __name__ == "__main__":
    main(config)